## Large-Scale JSON to JSONL Conversion
This notebook implements a memory-efficient pipeline to preprocess and convert large raw data files (specifically from the **OdiaGenAI** corpus) into a machine-learning-ready **JSONL (JSON Lines)** format.

### Key Achievements:
* **Memory-Efficient Streaming:**
  * Utilizes the `ijson` library to parse massive JSON files iteratively.
  * This "streaming" approach allows processing datasets that are larger than the available RAM, preventing Google Colab session crashes.
* **Data Cleaning & Normalization:**
  * Implements a `clean_text_fast` function to sanitize the raw text.
  * Removes non-standard whitespace (newlines, tabs) and collapses multiple spaces into single spaces to ensure tokenization consistency.
* **Format Transformation:**
  * Converts the hierarchical JSON structure into a flat **JSONL** format (one record per line), which is the standard input format for most Hugging Face training scripts and tokenizer training pipelines.
  * **Verified Output:** Successfully processed and saved 100,000 records (in the example run) to `.../varta_02_cleaned.jsonl`.

### Data Context
* **Source:** [OdiaGenAIdata/pre_train_odia_data_processed (Hugging Face)](https://huggingface.co/datasets/OdiaGenAIdata/pre_train_odia_data_processed).
* **Input File:** `varta_02.json` (Likely a subset or shard of the full corpus).
* **Output Format:** Cleaned JSONL with a single `"text"` field per line.

### Citation

```bibtex
@misc{Odia_LLM_Corpus,
  author = {Shantipriya Parida and Sambit Sekhar and Debasish Dhal and Pritiprava Mishra and Suman Kumar Maharana and Purushottam Kumar and Priyabrata Jena and Gunnet Singh Kohli and Kalyanamalini Sahoo},
  title = {Large Odia LLM Corpus},
  year = {2024},
  publisher = {Hugging Face},
  journal = {Hugging Face repository},
  howpublished = {\url{https://huggingface.co/OdiaGenAI}},
}
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 7.6 MB/s eta 0:00:00


In [ ]:
import json
import ijson
import re
import os
from tqdm import tqdm

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
input_file = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/varta_02.json"
output_file = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_cleaned.jsonl"

In [ ]:
# ==========================================
# HELPER FUNCTION: TEXT CLEANING
# ==========================================
def clean_text_fast(text):
    """
    Normalizes input text by sanitizing whitespace and control characters.

    This utility function performs three specific cleaning operations:
    1. Replaces newlines, tabs, and carriage returns with single spaces.
    2. Collapses multiple consecutive spaces into a single space.
    3. Strips leading and trailing whitespace.

    Args:
        text (str or None): The raw input string to be processed.

    Returns:
        str: The normalized string. Returns an empty string if the input
             is None or empty.
    """
    if not text:
        return ""
    # 1. Replace explicit newlines/tabs/returns with a single space
    text = text.replace('\n', ' ').replace('\t', ' ').replace('\r', ' ')

    # 2. Collapse multiple spaces (2 or more) into a single space
    # (Regex is slightly slower than replace, but necessary for '  ' -> ' ')
    text = re.sub(r' {2,}', ' ', text)

    # 3. Strip leading and trailing whitespace
    return text.strip()

In [ ]:
# ==========================================
# MAIN PROCESSING LOOP (STREAMING)
# ==========================================
print(f"Reading from: {input_file}")
print(f"Writing to:   {output_file}")
print("Processing...")

count = 0

# Open input in Binary Read mode ('rb') for ijson
# Open output in Text Write mode ('w')
with open(input_file, "rb") as infile, open(output_file, "w", encoding="utf-8") as outfile:

    # ijson.items yields objects one by one without loading the whole file into RAM
    # "item" assumes the JSON is a top-level array [ {}, {}, ... ]
    for record in tqdm(ijson.items(infile, "item"), desc="Records Processed"):

        # --- LOGIC TO FIX STRUCTURE (List vs Dict) ---
        obj = None

        # If the record is a list [{"text":...}], extract the first item
        if isinstance(record, list):
            if len(record) > 0:
                obj = record[0]
        # If the record is already a dictionary {"text":...}, use it
        elif isinstance(record, dict):
            obj = record

        # --- LOGIC TO CLEAN TEXT ---
        if obj and isinstance(obj, dict):
            if "text" in obj and isinstance(obj["text"], str):
                # Apply the cleaning function
                obj["text"] = clean_text_fast(obj["text"])

            # Write line to file (ensure_ascii=False preserves Odia script)
            outfile.write(json.dumps(obj, ensure_ascii=False) + "\n")
            count += 1

print(f"\n✅ Success! Processed {count} records.")
print(f"File saved to: {output_file}")

Reading from: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/varta_02.json
Writing to:   /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_cleaned.jsonl
Processing...


Records Processed: 100000it [00:16, 6084.15it/s]


✅ Success! Processed 100000 records.
File saved to: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/varta_02_cleaned.jsonl


In [ ]:
# ==========================================
# VERIFICATION
# ==========================================
print("\n--- Verifying First 3 Records ---")
with open(output_file, "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        if not line: break
        print(line.strip())


--- Verifying First 3 Records ---
{"text": "ଲକ୍ଷ୍ନୋ,୧୪।୨: ପ୍ରଥମ ପର୍ଯ୍ୟାୟ ପରେ ସୋମବାର ଉତ୍ତରପ୍ରଦେଶର ୫୫ ଆସନରେ ଦ୍ୱିତୀୟ ପର୍ଯ୍ୟାୟ ଭୋଟ ଗ୍ରହଣ ଚାଳିଛି। ମତଦାନ ସକାଳ ୭ଟାରୁ ଆରମ୍ଭ ହୋଇଥିବା ବେଳେ ଅପରାହ୍ନ ସାଢେ ୩ଟା ସୁଦ୍ଧା ୫୧.୯୩% ମତଦାନ ହୋଇଛି। ଦ୍ବିତୀୟ ପର୍ଯ୍ୟାୟରେ ୫୮୪ ପ୍ରାର୍ଥୀ ପ୍ରତିଦ୍ବନ୍ଦ୍ବିତା କରୁଛନ୍ତି। ସନ୍ଧ୍ୟା ୬ଟା ପର୍ଯ୍ୟନ୍ତ ମତଦାନ ହେବ। ବିରୋଧୀ ଭାଜପା ସରକାରଙ୍କ ନୀତିକୁ ସମାଲୋଚନା କରୁଥିବା ବେଳେ ଏହି ନିର୍ବାଚନକୁ ମୋଦି ସରକାରଙ୍କ ଲୋକପ୍ରିୟତାର ଅସଲ ଅଗ୍ନିପରୀକ୍ଷା ବୋଲି କୁହାଯାଉଛି।", "source": "varta"}
{"text": "ଚଣ୍ଡିଗଡ,୧୪ ।୨: ପଞ୍ଜାବ ବିଧାନସଭା ନିର୍ବାଚନ ଲାଗି ପ୍ରଧାନମନ୍ତ୍ରୀ ନରେନ୍ଦ୍ର ମୋଦି ସୋମବାର ପ୍ରଥମ କରି ସେଠାରେ ଏକ ଜନସଭାକୁ ସମ୍ବୋଧନ କରିଛନ୍ତି । ଜଳନ୍ଧରରେ ମୋଦି ଏକ ରାଲି କରିବା ସହ ଏକ ଜନସଭାରେ ଉଦବୋଧନ ଦେଇଛନ୍ତି । ମୋଦି କହିଛନ୍ତି ପଞ୍ଜାବ ମାଟି ସହ ମୋର ରହିଛି ପୁରୁଣା ସମ୍ପର୍କ । ମୋଦିଙ୍କ ଏହି ରାଲି ପାଇଁ ଜଳନ୍ଧରରେ କଡା ସୁରକ୍ଷା ବ୍ୟବସ୍ଥା କରାଯାଇଛି । ଯାହା ଫଳରେ ପଞ୍ଜାବ ମୁଖ୍ୟମନ୍ତ୍ରୀ ଚରଣଜିତ ସିଂ ତାଙ୍କ ରାଲିରେ ଯୋଗଦେବାକୁ ଯାଇ ପାରି ନ ଥିବା ଅଭିଯୋଗ ହୋଇଛି । ସୂଚନାଯୋଗ୍ୟ, ଜାନୁୟାରୀରେ ପ୍ରଧାନମନ୍ତ୍ରୀ ଯେତେବେଳେ ପଞ୍ଜାବ ଗସ୍ତରେ ଆସିଥିଲେ ସେତେବେଳେ ତାଙ୍କୁ ଅଧାବାଟରେ କେତେଜଣ ପ୍ରଦର୍ଶନକାରୀ ଅଟକାଇ ଦେଇଥିଲେ । 